<a href="https://colab.research.google.com/github/rainforest01-coder/ESAA_files/blob/OB/0306_%EC%84%B8%EC%85%98_%EB%B6%84%EB%A5%98_%EC%97%B0%EC%8A%B5%EB%AC%B8%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **분류 연습 문제**
___
출처 : 핸즈온 머신러닝 Ch03 분류 연습문제 1, 2번

In [1]:
# import data
from sklearn.datasets import fetch_openml
mnist = fetch_openml('mnist_784', version = 1, as_frame = False)

In [2]:
X, y = mnist["data"], mnist["target"]

In [3]:
X_train, X_test, y_train, y_test = X[:60000], X[60000:], y[:60000], y[60000:]

### **1. MNIST 데이터셋으로 분류기를 만들어 테스트 세트에서 97% 정확도를 달성해보세요.**
___

1. `KNeghtborsClassifier`를 사용하는 것을 추천합니다.
2. `weights`와 `n_neighbors` 하이퍼 파라미터로 그리드 탐색을 시도하여, 좋은 하이퍼 파라미터 값을 찾아보세요.

In [5]:
# import package
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
knc=KNeighborsClassifier()

In [6]:
# Try GridSearch to optimize hyperparameter
params={'weights':['uniform','distance'],'n_neighbors':[3,4,5]}
gscv=GridSearchCV(knc,param_grid=params,scoring='accuracy',cv=3,verbose=1)
gscv.fit(X_train,y_train)

Fitting 3 folds for each of 6 candidates, totalling 18 fits


GridSearchCV(cv=3, estimator=KNeighborsClassifier(),
             param_grid={'n_neighbors': [3, 4, 5],
                         'weights': ['uniform', 'distance']},
             scoring='accuracy', verbose=1)

In [7]:
# best hyperparameter
gscv.best_params_

{'n_neighbors': 4, 'weights': 'distance'}

In [8]:
# best score
gscv.best_score_

np.float64(0.9703500000000002)

In [9]:
# model test
y_pred=gscv.predict(X_test)

### **2. 다음 단계를 따라 인위적으로 훈련 세트를 늘리는 데이터 증식 또는 훈련 세트 확장 기법을 연습해봅시다.**
___

#### **STEP 1. MNIST 이미지를 (왼, 오른, 위, 아래) 어느 방향으로든 한 픽셀 이동시킬 수 있는 함수를 만들어 보세요.**

In [10]:
import numpy as np

def shift_mnist(image, direction):
    """
    image: (28, 28) numpy array
    direction: 'left', 'right', 'up', 'down'
    """
    shifted = np.zeros_like(image)

    if direction == 'left':
        shifted[:, :-1] = image[:, 1:]
    elif direction == 'right':
        shifted[:, 1:] = image[:, :-1]
    elif direction == 'up':
        shifted[:-1, :] = image[1:, :]
    elif direction == 'down':
        shifted[1:, :] = image[:-1, :]
    else:
        raise ValueError("direction must be 'left', 'right', 'up', or 'down'")

    return shifted

####  **STEP 2. 앞에서 만든 함수를 이용하여, 훈련 세트에 있는 각 이미지에 대해 네 개의 이동된 복사본(방향마다 한 개씩)을 만들어 훈련 세트에 추가하세요**

In [13]:
for image, label in zip(X_train, y_train):

    image = image.reshape(28, 28)

    X_train_augmented.append(image)
    y_train_augmented.append(label)

    X_train_augmented.append(shift_mnist(image, 'left'))
    y_train_augmented.append(label)

    X_train_augmented.append(shift_mnist(image, 'right'))
    y_train_augmented.append(label)

    X_train_augmented.append(shift_mnist(image, 'up'))
    y_train_augmented.append(label)

    X_train_augmented.append(shift_mnist(image, 'down'))
    y_train_augmented.append(label)

X_train_augmented = [img.reshape(-1) for img in X_train_augmented]

X_train_augmented = np.array(X_train_augmented)
y_train_augmented = np.array(y_train_augmented)

####  **STEP 3. 위에서 확장한 데이터셋을 이용하여, 1번 문제에서 찾은 최적 모델을 훈련시키고, 테스트 세트에서 정확도를 측정해보세요**

In [16]:
knc2=KNeighborsClassifier(**gscv.best_params_)
knc2.fit(X_train_augmented,y_train_augmented)

KNeighborsClassifier(n_neighbors=4, weights='distance')

In [17]:
accuracy =knc2.score(X_test, y_test)
print(accuracy)

0.9763
